# Classification Models — Basic
**Prerequisites:** Python basics, familiarity with NumPy and Pandas.

This notebook introduces classification in machine learning with three fundamental algorithms.
After this notebook, move to **Classification Models — Advanced** for ensemble methods and tuning.

## What is Classification?

Classification is a supervised learning task where the goal is to predict a **discrete class label**
for each input sample.

Examples:
- Email: spam or not spam?
- Medical image: tumour or healthy tissue?
- Loan application: approve or reject?

Unlike regression (which predicts a number), classification predicts **which category** a sample belongs to.

### Binary vs. Multiclass
- **Binary:** Two possible classes (e.g., income >50K or ≤50K)
- **Multiclass:** More than two classes (e.g., digit recognition: 0–9)

## Dataset: Adult Income

The Adult dataset from the 1994 US Census contains demographic information about ~48,000 individuals.
The task: predict whether a person earns **>$50K per year**.

Key features: age, education, occupation, hours-per-week, etc.
Target: `income` — binary (">50K" or "≤50K")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay,
                              classification_report)

## Loading and Cleaning the Data

In [ ]:
df = pd.read_csv('../data/adult.csv', na_values='?')
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])
df.dropna(inplace=True)
print(f"\nShape after dropping rows with missing values: {df.shape}")

## Exploratory Data Analysis

In [ ]:
# Class distribution — are the classes balanced?
income_counts = df['income'].value_counts()
print(income_counts)

plt.figure(figsize=(5, 4))
income_counts.plot(kind='bar', color=['steelblue', 'tomato'], rot=0)
plt.title('Class Distribution')
plt.ylabel('Count')
plt.xlabel('Income')
plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio: {income_counts.max() / income_counts.min():.1f}:1")

In [ ]:
# Age distribution by income
plt.figure(figsize=(8, 4))
for label, grp in df.groupby('income'):
    grp['age'].plot(kind='kde', label=label)
plt.title('Age Distribution by Income')
plt.xlabel('Age')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Hours worked per week vs income
plt.figure(figsize=(8, 4))
sns.boxplot(x='income', y='hours-per-week', data=df, palette=['steelblue', 'tomato'])
plt.title('Hours per Week by Income Group')
plt.show()

## Feature Engineering

Grouping education levels makes the model simpler and the data easier to understand.

In [ ]:
edu_dict = {
    'Preschool': 'Elementary', '1st-4th': 'Elementary', '5th-6th': 'Elementary',
    '7th-8th': 'Middle_School', '9th': 'Middle_School', '10th': 'Middle_School', '11th': 'Middle_School',
    '12th': 'High_School', 'HS-grad': 'High_School',
    'Some-college': 'Undergraduate', 'Bachelors': 'Undergraduate',
    'Assoc-voc': 'Undergraduate', 'Prof-school': 'Undergraduate',
    'Masters': 'Graduate', 'Assoc-acdm': 'Graduate', 'Doctorate': 'Graduate',
}
df['education'] = df['education'].replace(edu_dict)

plt.figure(figsize=(8, 4))
order = df.groupby('education')['income'].apply(lambda x: (x=='>50K').mean()).sort_values(ascending=False).index
sns.barplot(x='education', y=(df['income']=='>50K').astype(int),
            data=df, order=order, palette='Blues_r', errorbar=None)
plt.title('Proportion Earning >50K by Education Level')
plt.ylabel('Proportion >50K')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Preprocessing Pipeline

We need to handle two types of features differently:
- **Numeric features** (age, hours-per-week, etc.): Standardize with `StandardScaler`
- **Categorical features** (education, occupation, etc.): Encode with `OneHotEncoder`

`ColumnTransformer` applies different transformations to different columns simultaneously.

**Important:** All preprocessing is wrapped in a `Pipeline` so it fits *inside* cross-validation — preventing data leakage.

In [ ]:
feature_cols = ['age', 'workclass', 'education', 'educational-num', 'marital-status',
                'occupation', 'relationship', 'race', 'gender',
                'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

X = df[feature_cols]
y = (df['income'] == '>50K').astype(int)

print(f"Features: {X.shape[1]}")
print(f"Positive class (>50K): {y.mean():.1%}")

# Stratified split preserves class ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain size: {len(X_train)}  |  Test size: {len(X_test)}")

In [ ]:
numeric_features = ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical_features = ['workclass', 'education', 'marital-status', 'occupation',
                        'relationship', 'race', 'gender', 'native-country']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
])

## Evaluation Metrics

For classification we track:
- **Accuracy:** Overall correct predictions (can be misleading when classes are imbalanced)
- **Precision:** Of predicted positives, how many are actually positive? (avoid false alarms)
- **Recall:** Of actual positives, how many did we catch? (avoid missing cases)
- **F1 Score:** Harmonic mean of Precision and Recall (balanced when both matter)

We'll also plot a **Confusion Matrix** after each model.

In [ ]:
def evaluate_clf(name, y_true, y_pred, show_matrix=True):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    print(f"\n{'─'*45}  {name}")
    print(f"  Accuracy:  {acc:.4f}  |  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}  |  F1 Score:  {f1:.4f}")
    if show_matrix:
        fig, ax = plt.subplots(figsize=(4, 3))
        ConfusionMatrixDisplay.from_predictions(
            y_true, y_pred, display_labels=['≤50K', '>50K'], cmap='Blues', ax=ax)
        ax.set_title(f'Confusion Matrix — {name}')
        plt.tight_layout()
        plt.show()
    return {'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1': f1}

## Logistic Regression

Despite the name, Logistic Regression is a **classification** algorithm. It models the probability
that a sample belongs to the positive class using the sigmoid function:

$$P(y=1) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \cdots)}}$$

If P > 0.5 → predict positive; otherwise predict negative.

**Strengths:**
- Fast and interpretable — coefficients show feature importance
- Good baseline for binary classification
- Works well when the decision boundary is approximately linear

**Weaknesses:** Assumes linear separability in feature space (after transformation).

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_lr = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
clf_lr.fit(X_train, y_train)
y_pred_lr = clf_lr.predict(X_test)

results = [evaluate_clf("Logistic Regression", y_test, y_pred_lr)]

In [ ]:
# Which features matter most? (after scaling, coefficients are comparable)
lr_model = clf_lr.named_steps['classifier']
ohe_names = clf_lr.named_steps['preprocessor'].transformers_[1][1].get_feature_names_out(categorical_features)
all_names = numeric_features + list(ohe_names)

coef_series = pd.Series(lr_model.coef_[0], index=all_names)
top = coef_series.abs().nlargest(12).index
coef_series[top].sort_values().plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Logistic Regression — Top 12 Feature Coefficients')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

## Decision Tree

A Decision Tree learns a sequence of binary rules (e.g., `marital-status = Married → go left`),
forming a tree structure from root to leaves where each leaf predicts a class.

**Strengths:**
- Very interpretable — you can follow the exact decision path
- No scaling required
- Naturally handles categorical features

**Weaknesses:** Without depth limits it **overfits** easily (memorises training data).

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

clf_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(max_depth=5, random_state=42))
])
clf_dt.fit(X_train, y_train)
y_pred_dt = clf_dt.predict(X_test)

results.append(evaluate_clf("Decision Tree (max_depth=5)", y_test, y_pred_dt))

In [ ]:
# Show the depth vs. overfitting tradeoff
depths = range(1, 15)
train_f1, test_f1 = [], []
for d in depths:
    m = Pipeline([('pre', preprocessor),
                  ('clf', DecisionTreeClassifier(max_depth=d, random_state=42))])
    m.fit(X_train, y_train)
    train_f1.append(f1_score(y_train, m.predict(X_train)))
    test_f1.append(f1_score(y_test,  m.predict(X_test)))

plt.figure(figsize=(8, 4))
plt.plot(depths, train_f1, 'o-', label='Train F1', color='steelblue')
plt.plot(depths, test_f1,  'o-', label='Test F1',  color='tomato')
plt.xlabel('max_depth')
plt.ylabel('F1 Score')
plt.title('Decision Tree — Overfitting as Depth Increases')
plt.legend()
plt.grid(True)
plt.show()

## K-Nearest Neighbors (KNN)

KNN makes predictions based on the majority class among the **k most similar training samples**.
"Similar" is measured by Euclidean distance in the feature space.

$$d(a, b) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

**Strengths:**
- Intuitive — "you are like your neighbours"
- No training phase (lazy learner)
- Naturally captures non-linear boundaries

**Weaknesses:**
- Slow at prediction time (must compare against all training samples)
- Sensitive to feature scale → **always scale**
- Struggles with high-dimensional data (curse of dimensionality)
- Choice of k matters a lot

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

clf_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier(n_neighbors=11))
])
clf_knn.fit(X_train, y_train)
y_pred_knn = clf_knn.predict(X_test)

results.append(evaluate_clf("KNN (k=11)", y_test, y_pred_knn))

In [ ]:
# Finding the best k via test F1
k_values = range(1, 31)
knn_f1 = []
for k in k_values:
    m = Pipeline([('pre', preprocessor),
                  ('clf', KNeighborsClassifier(n_neighbors=k))])
    m.fit(X_train, y_train)
    knn_f1.append(f1_score(y_test, m.predict(X_test)))

best_k = k_values[np.argmax(knn_f1)]
plt.figure(figsize=(8, 4))
plt.plot(k_values, knn_f1, 'o-', color='steelblue')
plt.axvline(best_k, color='tomato', linestyle='--', label=f'Best k={best_k}')
plt.xlabel('k (Number of Neighbours)')
plt.ylabel('F1 Score')
plt.title('KNN — Choosing k')
plt.legend()
plt.grid(True)
plt.show()

## Comparison: Basic Models

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print(results_df.round(4))

results_df.plot(kind='bar', figsize=(10, 5), rot=10)
plt.title('Basic Classification Models — Comparison')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

## Summary

| Model | Key Idea | Needs Scaling | Interpretable |
|-------|----------|---------------|---------------|
| Logistic Regression | Probability via sigmoid | Yes | Yes (coefficients) |
| Decision Tree | Binary rules from root to leaf | No | Yes (visualisable) |
| KNN | Majority vote of k neighbours | Yes | Somewhat |

**Next steps:** Head to **Classification Models — Advanced** for Random Forest, SVM, XGBoost, and full pipeline comparison.